In [ ]:
import pandas as pd
import plotly.graph_objects as go
from clickhouse_driver import Client

# Connect to ClickHouse
client = Client('localhost')

# SQL query to get hiring funnel data
query = """
        WITH funnel AS (SELECT 'Applied' as stage, count(*) as count
        FROM hr.applications

        UNION ALL

        SELECT 'Phone Screen' as stage, count(*) as count
        FROM hr.applications
        WHERE phone_screen_date IS NOT NULL

        UNION ALL

        SELECT 'Interview' as stage, count(*) as count
        FROM hr.applications
        WHERE interview_date IS NOT NULL

        UNION ALL

        SELECT 'Offer' as stage, count(*) as count
        FROM hr.applications
        WHERE offer_date IS NOT NULL

        UNION ALL

        SELECT 'Hired' as stage, count(*) as count
        FROM hr.applications
        WHERE hire_date IS NOT NULL )
        SELECT stage, count, count * 100.0 / first_value(count) OVER (ORDER BY count DESC) as conversion_rate
        FROM funnel
        ORDER BY count DESC; \
        """

# Get data from ClickHouse
df = pd.DataFrame(client.execute(query), columns=['Stage', 'Count', 'Conversion_Rate'])

# Create funnel visualization
fig = go.Figure(go.Funnel(
	y=df['Stage'],
	x=df['Conversion_Rate'],
	textinfo="value+percent initial",
	textposition="inside",
	opacity=0.65,
	marker={
		"color": ["#003f5c", "#58508d", "#bc5090", "#ff6361", "#ffa600"],
		"line": {"width": 1, "color": ["white"]}
	},
))

fig.update_layout(
	title="Hiring Funnel Analysis",
	yaxis_title="Hiring Stage",
	xaxis_title="Conversion Rate (%)",
	showlegend=False
)

fig.show()
